# Retrieval study results

Select one dataset and sample size below. This notebook reads saved run summaries; it does not rerun retrieval or inference. Compare runs only when their selected question IDs match.


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

OUTPUTS = Path("../outputs")
DATASET = "hotpotqa"  # "hotpotqa" or "musique"
SAMPLE_SIZE = 100


In [ ]:
rows = []
samples = {}
for folder in sorted(OUTPUTS.iterdir()):
    metadata_path = folder / "run_metadata.json"
    summary_path = folder / "summary.json"
    if not metadata_path.is_file() or not summary_path.is_file():
        continue
    metadata = json.loads(metadata_path.read_text())
    if metadata.get("scope") != "paragraph_retrieval_study":
        continue
    dataset_name = metadata["config"]["dataset"]["dataset_name"].lower()
    if DATASET not in dataset_name or len(metadata.get("selected_question_ids", [])) != SAMPLE_SIZE:
        continue
    summary = json.loads(summary_path.read_text())
    samples[folder.name] = tuple(metadata["selected_question_ids"])
    rows.append({
        "run": folder.name,
        "gold %": summary["gold_evidence_percentage"]["mean"],
        "full recovery %": 100 * summary["full_gold_recovery_rate"],
        "noise %": summary["noise_percentage"]["mean"],
        "paragraphs": summary["unique_paragraphs_retrieved"]["mean"],
        "rounds": summary["retrieval_rounds"]["mean"],
        "errors": summary["errors"],
    })
table = pd.DataFrame(rows).set_index("run") if rows else pd.DataFrame()
display(table.round(2))


In [ ]:
# A paired comparison needs the same questions in the same order.
groups = {}
for run, ids in samples.items():
    groups.setdefault(ids, []).append(run)
print(f"{len(groups)} distinct ordered question samples")
for i, runs in enumerate(groups.values(), 1):
    print("Sample", i, ":", len(runs), "runs:", ", ".join(runs))


In [ ]:
if not table.empty:
    ax = table[["gold %", "noise %", "full recovery %"]].plot.bar(figsize=(12, 5))
    ax.set_ylabel("Percent")
    ax.set_title(f"{DATASET} — {SAMPLE_SIZE} questions per run")
    ax.legend(loc="best")
    plt.xticks(rotation=40, ha="right")
    plt.tight_layout()
    plt.show()
